In [82]:
import importlib

In [83]:


import pandas as pd
import data_loader
import user_profile as user_pf


In [84]:
# Change this to True if you are running on Google Colab
RUNNING_ON_COLAB = False

# Constants
USER_ID = 999999

In [85]:
if RUNNING_ON_COLAB:
    data_loader.mount_drive()

In [86]:

films_df = data_loader.load_movies()

In [87]:
# Load the data
# films_df = data_loader.load_movies()
ratings_df = data_loader.load_ratings()
credits_df = data_loader.load_credits()

In [88]:
print("Data dimensions:")
print("films_df: ", films_df.shape)
print("ratings_df: ", ratings_df.shape)
print("credits_df: ", credits_df.shape)

Data dimensions:
films_df:  (1070323, 8)
ratings_df:  (32000204, 3)
credits_df:  (45476, 3)


In [89]:
credits_df.shape

(45476, 3)

In [90]:
films_df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'release_date', 'imdb_id',
       'popularity', 'genres'],
      dtype='object')

In [91]:
# Find film_df entry 100 - Lock Stock and Two Smoking Barrels
films_df[films_df['id'] == 100]

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,genres
65,100,"Lock, Stock and Two Smoking Barrels",8.115,6692.0,1998-08-28,tt0120735,1.946,"Comedy, Crime"


In [92]:
# Drop the weird film entry
films_df = films_df.drop(35587)

In [93]:
import preprocessing as pre
# importlib.reload(pre)

In [94]:
# Drop films before 1985 and after today
films_df = pre.filter_films(films_df)


In [114]:
films_df.shape

(28833, 28)

In [96]:
films_df.sort_values('release_date', ascending=False)

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,genres
304132,447273,Snow White,2.5,22.0,2025-03-19,tt6208148,24.058,"Family, Fantasy"
939885,1297763,Batman Ninja vs. Yakuza League,5.7,22.0,2025-03-17,tt32508210,18.282,"Animation, Action"
1059269,1438267,Gamad Machmad 4,4.4,8.0,2025-03-16,NaN,10.171,"Drama, Comedy, Thriller, Action"
930641,1286773,The Metropolitan Opera: Fidelio,7.4,7.0,2025-03-15,NaN,11.995,Music
1042445,1417677,Bill Burr: Drop Dead Years,7.9,15.0,2025-03-14,tt34686029,6.094,Comedy
...,...,...,...,...,...,...,...,...
141908,248595,Alien Outlaw,2.7,13.0,1985-01-01,tt0190229,3.308,"Mystery, Science Fiction, Comedy, Western, Horror"
59004,82942,'Master Harold'... and the Boys,6.2,6.0,1985-01-01,tt0089564,4.057,Drama
127434,220343,Anna Karenina,5.2,6.0,1985-01-01,tt0088726,4.618,"Drama, Romance, TV Movie"
98676,161426,You Killed Me First,4.8,19.0,1985-01-01,tt0190157,0.671,"Drama, Horror"


In [97]:
films_df.head()

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,genres
0,2,Ariel,7.1,340.0,1988-10-21,tt0094675,9.400,"Comedy, Drama, Romance, Crime"
1,3,Shadows in Paradise,7.3,403.0,1986-10-17,tt0092149,7.088,"Comedy, Drama, Romance"
2,5,Four Rooms,5.9,2678.0,1995-12-09,tt0113101,3.547,"Comedy, Crime"
3,6,Judgment Night,6.5,333.0,1993-10-15,tt0107286,12.110,"Action, Crime, Thriller"
4,8,Life in Loops (A Megacities RMX),7.5,27.0,2006-01-01,tt0825671,3.203,Documentary


In [98]:
genres_films = films_df.copy()


In [99]:
# Data preprocessing
# # One-hot encode the genres
ohe_films_df, genre_list_mlb = pre.one_hot_encode_genres(genres_films)

In [100]:
ohe_films_df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'release_date', 'imdb_id',
       'popularity', 'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror',
       'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie',
       'Thriller', 'War', 'Western'],
      dtype='object')

In [101]:
# Gather info on directors and cast
credits_df = pre.condense_credits(credits_df)

In [102]:
print(type(ohe_films_df), type(credits_df))  # Debugging line

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>


In [103]:

# Tidy the noise and merge the credits' metadata with the films DataFrame
films_df = pre.data_tidying(ohe_films_df, credits_df)
# print(films_df.columns)


In [104]:
# Generate the user profile
user_ratings_df = user_pf.load_user_ratings()
user_profile = user_pf.create_user_profile(USER_ID, films_df, user_ratings_df, genre_list_mlb)

In [105]:
# Append the user profile to the ratings DataFrame
ratings_df = pd.concat([ratings_df, user_ratings_df], ignore_index=True)

In [106]:
# print all the values for vote_average in films_df
print((films_df['vote_count'] > 100).sum())

11368


In [107]:
films_df[films_df['vote_average'] != 0]

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,Action,Adventure,Animation,...,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western,cast_info,director_info
0,2,Ariel,7.100,340.0,1988-10-21,tt0094675,9.400,0,0,0,...,0,0,1,0,0,0,0,0,"[(Turo Pajala, 54768), (Susanna Haavisto, 5476...","(Aki Kaurismäki, 2)"
1,3,Shadows in Paradise,7.300,403.0,1986-10-17,tt0092149,7.088,0,0,0,...,0,0,1,0,0,0,0,0,"[(Matti Pellonpää, 4826), (Kati Outinen, 5999)...","(Aki Kaurismäki, 3)"
2,5,Four Rooms,5.900,2678.0,1995-12-09,tt0113101,3.547,0,0,0,...,0,0,0,0,0,0,0,0,"[(Tim Roth, 3129), (Antonio Banderas, 3131), (...","(Allison Anders, 5)"
3,6,Judgment Night,6.500,333.0,1993-10-15,tt0107286,12.110,1,0,0,...,0,0,0,0,0,1,0,0,"[(Emilio Estevez, 2880), (Cuba Gooding Jr., 97...","(Stephen Hopkins, 6)"
4,12,Finding Nemo,7.800,19500.0,2003-05-30,tt0266543,8.524,0,0,1,...,0,0,0,0,0,0,0,0,"[(Albert Brooks, 13), (Ellen DeGeneres, 14), (...","(Andrew Stanton, 12)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28828,463800,Firebase,6.800,145.0,2017-06-28,tt7078926,6.090,1,0,0,...,0,0,0,1,0,0,1,0,"[(Steve Boyle, 190921), (Nic Rhind, 964251), (...","(Neill Blomkamp, 463800)"
28829,463906,The Saint,5.300,280.0,2017-07-11,tt2569088,9.959,1,1,0,...,0,0,0,0,0,0,0,0,"[(Adam Rayner, 144292), (Eliza Dushku, 13446),...","(Ernie Barbarash, 463906)"
28830,464111,Zygote,7.000,167.0,2017-07-12,tt7078780,7.634,0,0,0,...,0,0,0,1,0,0,0,0,"[(Dakota Fanning, 501), (Jose Pablo Cantillo, ...","(Neill Blomkamp, 464111)"
28831,464207,The Truth Is in the Stars,7.067,15.0,2017-05-01,tt7104950,3.632,0,0,0,...,0,0,0,0,0,0,0,0,"[(William Shatner, 1748), (Neil deGrasse Tyson...","(Craig Thompson, 464207)"


In [125]:
import hybrid_recommender as hyb
importlib.reload(hyb)

<module 'hybrid_recommender' from '/Users/Cathal/Rec-Genie/rec-sys/hybrid_recommender.py'>

In [126]:
# Generate recommendations
recommendations = hyb.hybrid_recommend(999999, user_profile, films_df, credits_df, ratings_df, genre_list_mlb)

User-User algorithm set up!
Rating before: 1.6435
Rating after: -0.35650000000000004
Rating before: 1.55
Rating after: -0.44999999999999996
Rating before: 1.95
Rating after: -0.050000000000000044
Rating before: 1.95
Rating after: -0.050000000000000044
Rating before: 1.95
Rating after: -0.050000000000000044
Rating before: 1.95
Rating after: -0.050000000000000044
Rating before: 1.9
Rating after: -0.10000000000000009
Rating before: 1.75
Rating after: -0.25
Rating before: 1.8
Rating after: -0.19999999999999996
Rating before: 1.65
Rating after: -0.3500000000000001
Rating before: 1.8
Rating after: -0.19999999999999996
Rating before: 1.66
Rating after: -0.3400000000000001
Rating before: 1.55
Rating after: -0.44999999999999996
Rating before: 1.75
Rating after: -0.25
Rating before: 1.75
Rating after: -0.25
Rating before: 1.95
Rating after: -0.050000000000000044
Rating before: 1.95
Rating after: -0.050000000000000044
Rating before: 1.8
Rating after: -0.19999999999999996
Rating before: 1.95
Ratin

In [110]:
len(recommendations)

400

In [123]:
score_breakdown = hyb.score_breakdown(films_df, recommendations)

In [124]:
score_breakdown.head(50)

,id,title,release_date,vote_average,vote_count,score,cast_score,director_score,genre_score,user_user_score
143,272,Batman Begins,2005-06-10,7.700,21297.0,12.571658,5.40,4.88,7.4620,3.286298
775,1894,Star Wars: Episode II - Attack of the Clones,2002-05-15,6.600,13441.0,12.434603,7.98,2.92,5.9840,3.129083
569,1420,Breakfast on Pluto,2005-11-16,7.200,396.0,9.970016,4.08,2.00,7.4190,3.636696
1045,2567,The Aviator,2004-12-17,7.221,5405.0,9.663816,3.63,1.80,9.8580,3.102576
84,155,The Dark Knight,2008-07-16,8.519,33561.0,9.320366,1.32,4.88,6.3795,3.194106
1309,4538,The Darjeeling Limited,2007-09-07,7.200,3621.0,9.149036,1.41,3.72,6.9660,3.607556
490,1124,The Prestige,2006-10-17,8.202,16363.0,9.070780,1.32,4.88,4.7220,3.408620
986,2288,Closer,2004-12-03,6.800,3460.0,8.954535,3.36,1.56,4.9830,4.115295
340,640,Catch Me If You Can,2002-12-16,7.978,15936.0,8.849514,3.60,1.92,7.1910,2.972034
495,1164,Babel,2006-10-26,7.200,3782.0,8.734677,3.21,0.00,9.8580,3.727437


In [113]:
# weights = {
#     'cast_ft_weight': 0.3,
#     'director_ft_weight': 0.3,
#     'genre_ft_weight': 0.15,
#     'user_user_weight': 1,
#     'content_weight': 0.7,
#     'collab_weight': 0.3
# }